### Procesamiento de Lenguaje Natural I

**Estudiante:** a2517 - Gabriela Di Grazia

**Docente:** Josselyn Ordóñez

# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [39]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Resolución del Desafío 1

Se mantiene el `TfidfVectorizer` del tutorial en default. Los hiperparámetros se modifican en el punto 3.

### 1. Similaridad entre documentos

Se toman 5 documentos al azar del conjunto de entrenamiento (semilla 42). Se excluyen los que quedan vacíos al remover headers, footers y quotes.

Para cada uno se mide la similaridad coseno con el resto de train, como en el ejemplo del documento 4811, y se obtienen los 5 más similares.

Además se listan las palabras con mayor peso TF-IDF del documento y de cada vecino, para ver por qué el vectorizador los considera similares.


In [24]:
def terminos_top(idx, k=8):
    fila = X_train[idx].tocoo()
    pares = sorted(zip(fila.data, fila.col), reverse=True)[:k]
    return [idx2word[col] for val, col in pares]


rng = np.random.default_rng(42)
candidatos = [i for i, t in enumerate(newsgroups_train.data) if len(t.strip()) > 80]
idxs = rng.choice(candidatos, size=5, replace=False)
print("Documentos al azar:", idxs)
print()

for n, idx in enumerate(idxs, 1):
    print("=" * 90)
    print(f"Documento {n} de 5 — índice {idx}")
    print(newsgroups_train.data[idx][:500])
    print()
    print("Clase:", newsgroups_train.target_names[y_train[idx]])
    print("Términos con más peso TF-IDF:", ", ".join(terminos_top(idx)))

    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]

    print()
    print("5 documentos más similares:")
    for i in mostsim:
        misma = y_train[i] == y_train[idx]
        comunes = set(terminos_top(idx)) & set(terminos_top(i))
        print(f"  sim={cossim[i]:.3f} | {newsgroups_train.target_names[y_train[i]]} | misma etiqueta: {misma}")
        print("  top del vecino:", ", ".join(terminos_top(i)))
        print(
            "  palabras en común (entre las de más peso):",
            ", ".join(sorted(comunes)) if comunes else "(ninguna de las top)",
        )
        print(" ", " ".join(newsgroups_train.data[i].split())[:220])
        print()


Documentos al azar: [8769 4979 7435 1011 4912]

Documento 1 de 5 — índice 8769
  > Hi.
  > 
  > I use Emacs and I want to customize my keyboard better.
  > When I set up stuff in my .emacs with a keymap and define-keys,
  > I can only access certain of the keys on my X-Terminal's
  > keyboard. I can't get e.g. F10, Home, End, PgUp, PgDn; they all
  > seem to have either the same or no keycode. I have a feeling
  > this can't be fixed in emacs itself, but that I need to do some
  > xmodmap stuff. Can someone help me?

Unfortunately, the key event handling is pretty much har

Clase: comp.windows.x
Términos con más peso TF-IDF: emacs, keys, the, code, sequences, source, patches, key

5 documentos más similares:
  sim=0.378 | comp.windows.x | misma etiqueta: True
  top del vecino: emacs, pageup, xmodmap, shift, table, hp, work, customised
  palabras en común (entre las de más peso): emacs
  It is actually worse than you think. I have the same problem, and have given up. Emacs has an intern


5 documentos más similares:
  sim=0.466 | sci.space | misma etiqueta: True
  top del vecino: overhead, university, that, wrap, resupply, did, reston, the
  palabras en común (entre las de más peso): reston, that, the
  I don't care who told you this it is not generally true. I see EVERY single line item on a contract and I have to sign it. There is no such thing as wrap at this university. I also asked around here. Ther is no wrap at M

  sim=0.463 | sci.space | misma etiqueta: True
  top del vecino: dennis, the, fee, nasa, centers, reston, you, overhead
  palabras en común (entre las de más peso): reston, the, you
  Dennis, I have worked on or written proposals worth tens of millions of $$. Customers included government (including NASA), for profit and non-profit companies. All expected a wrap (usually called a fee). Much of the wor

  sim=0.424 | talk.politics.mideast | misma etiqueta: False
  top del vecino: the, and, they, were, was, that, he, to
  palabras en común (entre las de 

**Interpretación**

El vectorizador se dejó en default: no elimina stopwords. Eso se observa en varios vecinos.

**Documento 8769 (`comp.windows.x`) — 3/5 de la misma clase.**  
El texto consulta el mapeo de teclas de Emacs en una X-Terminal. Los términos de mayor peso son `emacs`, `keys`, `key`, `shift`. Los dos vecinos más cercanos también hablan de Emacs y teclas (`xmodmap`, `xterm`, `keymap`): la similaridad coincide con el contenido y con la etiqueta. El tercero es una FAQ de X; las palabras en común son `the` y `to`, que no aportan tema. Los dos documentos de `sci.crypt` se acercan por `key`/`keys`. En criptografía *key* es clave, no tecla. El modelo no distingue sentidos: si el peso TF-IDF de `key` es alto en ambos, los junta.

Conclusión: la similaridad funciona cuando hay un término de dominio (`emacs`) y falla cuando el término es ambiguo (`key`).

**Documento 4979 (`talk.politics.misc`) — 1/5.**  
El texto es un debate sobre aborto (`abortion`, `pro`, `choicers`, `clinic`). El único vecino de la misma clase habla de `fetus` / `womb`, coherente con el tema. Los demás pertenecen a religión o política internacional y, entre las palabras de más peso, solo se comparte `you`. Con el vectorizador en default un pronombre puede acercar documentos que no tratan el mismo tema.

Conclusión: en este caso la similaridad no se explica por el contenido. Se explica por el estilo (mucho *you*) y por no filtrar stopwords.

**Documento 7435 (`rec.autos`) — 4/5.**  
El post pide opciones poco útiles de un auto. Pesan `options`, `car`, `pointless` y también `regards`. Los dos vecinos más cercanos son el hilo inverso (opciones útiles: `tripmeter`, `engine`); los une `options` y la etiqueta coincide. Otro vecino habla de `car`, `loan` y `dealer`, también razonable. El documento de `misc.forsale` es casi solo “Regards, Steve.”: el top es `regards` y `steve`. Los une la firma, no el auto.

Conclusión: con un documento corto, una palabra de cortesía puede dominar el coseno. El 4/5 es coherente; el error viene de un post sin contenido.

**Documento 1011 (`sci.space`) — 2/5.**  
El texto discute el *wrap* (overhead) en contratos NASA (`allen`, `reston`, `jsc`, `wrap`). Los dos primeros vecinos son el mismo hilo (`wrap`, `reston`, `overhead`): contenido y etiqueta coinciden. Después aparecen un texto largo de Medio Oriente, la FAQ de ateísmo y un discurso presidencial, con similaridad ~0.41. Las palabras en común son `the`, `that`, `and`, `of`, `you`. No aparecen `nasa` ni `wrap`.

Conclusión: un documento largo con muchas stopwords puede obtener un coseno alto sin compartir el tema. El default no está midiendo el contenido en este caso.

**Documento 4912 (`misc.forsale`) — 2/5 en etiqueta, coherente en contenido.**  
Se vende un módem fax 14.4k. El top incluye `fax`, `zoom`, `32bis`. Los vecinos preguntan por fax o módem en grupos de hardware; casi todos comparten `fax` o `modem`. Solo dos tienen etiqueta `misc.forsale`.

Conclusión: TF-IDF junta bien el producto. No puede distinguir “vendo” de “busco”, porque esa diferencia está en la etiqueta, no en las palabras técnicas. No se interpreta como un fallo de similaridad de contenido.

**Conclusión.**  
La similaridad resulta razonable cuando hay vocabulario de dominio (`emacs`, `options`/`car`, `fax`/`modem`, `wrap`/`reston`). Falla cuando ganan stopwords (`you`, `the`, `that`) o un término ambiguo (`key`). Mantener el default permite observar ese efecto; en el punto 3 se prueban stopwords sobre la clasificación.


### 2. Clasificación por prototipos (tipo zero-shot)

Se clasifica cada documento de test comparándolo con todos los de train mediante similaridad coseno. Se le asigna la **etiqueta de clase** (el newsgroup) del documento de entrenamiento más cercano.

Se reutiliza el TF-IDF default del tutorial para comparar el F1-macro con el Naïve Bayes del enunciado (0.585).


In [25]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

cossim = cosine_similarity(X_test, X_train)
mostsim = cossim.argmax(axis=1)
y_pred_proto = y_train[mostsim]

print("F1-macro prototipos:", f1_score(y_test, y_pred_proto, average="macro"))
print("F1-macro MultinomialNB del tutorial:", f1_score(y_test, y_pred, average="macro"))
print("Tests con similaridad máxima 0:", (cossim.max(axis=1) == 0).sum())


F1-macro prototipos: 0.5049911553681621
F1-macro MultinomialNB del tutorial: 0.5854345727938506
Tests con similaridad máxima 0: 224


**Interpretación**

El F1-macro del clasificador por prototipos es **0.505**, inferior al `MultinomialNB` del enunciado (**0.585**).

En el punto 1 se observó que el documento más cercano no siempre comparte etiqueta: a veces se junta por `key`, por `you` o por un texto muy largo. En este modelo esa única coincidencia define la clase de todo el test. Si el vecino está mal, la predicción está mal.

Además hay **224** documentos de test con similaridad máxima 0 (vacíos o con términos fuera del vocabulario). El modelo igual les asigna un “más cercano”, que en la práctica es arbitrario: esos 224 son error.

El valor 0.505 no es azar (con 20 clases el azar sería ~0.05). Cuando el post de test es casi el mismo hilo que uno de train, el prototipo acierta, como en los casos de Emacs y del *wrap* de NASA.

**Conclusión.** El modelo cumple la consigna y sirve como referencia. Rinde peor que Naïve Bayes porque NB usa todas las palabras para cada clase, mientras que el prototipo se apoya en un solo documento. Con el vectorizador en default, ese documento a veces se parece por las razones incorrectas.


### 3. Naïve Bayes: hiperparámetros y F1-macro

Se parte del modelo del enunciado (`TfidfVectorizer()` + `MultinomialNB()`, F1-macro 0.585) y se modifican hiperparámetros del vectorizador y del clasificador. Se comparan **MultinomialNB** y **ComplementNB**. No se cambia `ngram_range`.

El objetivo es interpretar por qué cada cambio modifica el F1-macro.


In [26]:
def evaluar(nombre, vectorizador, modelo):
    Xtr = vectorizador.fit_transform(newsgroups_train.data)
    Xte = vectorizador.transform(newsgroups_test.data)
    modelo.fit(Xtr, y_train)
    f1 = f1_score(y_test, modelo.predict(Xte), average="macro")
    print(f"{nombre:75s}  F1={f1:.4f}  vocab={Xtr.shape[1]}")
    return f1


experimentos = [
    ("TF-IDF default + MultinomialNB (baseline tutorial)", TfidfVectorizer(), MultinomialNB()),
    ("TF-IDF stop_words + MultinomialNB", TfidfVectorizer(stop_words="english"), MultinomialNB()),
    (
        "TF-IDF stop + min_df=2 + max_df=0.5 + sublinear + MultinomialNB",
        TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True),
        MultinomialNB(),
    ),
    (
        "igual + MultinomialNB(alpha=0.1)",
        TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True),
        MultinomialNB(alpha=0.1),
    ),
    (
        "igual + MultinomialNB(alpha=0.01)",
        TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True),
        MultinomialNB(alpha=0.01),
    ),
    (
        "igual + ComplementNB",
        TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True),
        ComplementNB(),
    ),
    (
        "igual + ComplementNB(alpha=0.1)",
        TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True),
        ComplementNB(alpha=0.1),
    ),
    (
        "igual + ComplementNB(alpha=0.01)",
        TfidfVectorizer(stop_words="english", min_df=2, max_df=0.5, sublinear_tf=True),
        ComplementNB(alpha=0.01),
    ),
    (
        "CountVectorizer stop + min_df=2 + ComplementNB(alpha=0.1)",
        CountVectorizer(stop_words="english", min_df=2, max_df=0.5),
        ComplementNB(alpha=0.1),
    ),
]

resultados = [(nombre, evaluar(nombre, vec, clf)) for nombre, vec, clf in experimentos]
mejor_nombre, mejor_f1 = max(resultados, key=lambda x: x[1])
print("\nMejor configuración:", mejor_nombre)
print(f"Mejor F1-macro: {mejor_f1:.4f}")


TF-IDF default + MultinomialNB (baseline tutorial)                           F1=0.5854  vocab=101631


TF-IDF stop_words + MultinomialNB                                            F1=0.6468  vocab=101322


TF-IDF stop + min_df=2 + max_df=0.5 + sublinear + MultinomialNB              F1=0.6437  vocab=39115


igual + MultinomialNB(alpha=0.1)                                             F1=0.6777  vocab=39115


igual + MultinomialNB(alpha=0.01)                                            F1=0.6731  vocab=39115


igual + ComplementNB                                                         F1=0.6921  vocab=39115


igual + ComplementNB(alpha=0.1)                                              F1=0.6878  vocab=39115


igual + ComplementNB(alpha=0.01)                                             F1=0.6730  vocab=39115


CountVectorizer stop + min_df=2 + ComplementNB(alpha=0.1)                    F1=0.6387  vocab=39115

Mejor configuración: igual + ComplementNB
Mejor F1-macro: 0.6921


**Interpretación**

| Configuración | F1-macro | Vocabulario |
|---|---|---|
| Default del enunciado (MultinomialNB) | 0.585 | 101.631 |
| `stop_words='english'` | 0.647 | 101.322 |
| stop + `min_df=2` + `max_df=0.5` + `sublinear_tf` + Multinomial | 0.644 | 39.115 |
| igual + `alpha=0.1` (Multinomial) | 0.678 | 39.115 |
| igual + `alpha=0.01` (Multinomial) | 0.673 | 39.115 |
| igual + **ComplementNB** (alpha=1) | **0.692** | 39.115 |
| ComplementNB `alpha=0.1` | 0.688 | 39.115 |
| ComplementNB `alpha=0.01` | 0.673 | 39.115 |
| `CountVectorizer` + ComplementNB | 0.639 | 39.115 |

**Stopwords (0.585 → 0.647).** Es el mayor incremento. Coincide con el punto 1: `you`, `the` y `that` acercaban documentos de distinto tema. Al eliminarlas, dejan de votar por igual en las 20 clases. El promedio macro pesa cada clase de la misma forma, así que reducir esa confusión (por ejemplo política vs religión) sube el F1.

**`min_df` y `max_df` (0.647 → 0.644).** El vocabulario pasa de 101k a 39k (términos que aparecen una sola vez o en casi todos los documentos). El F1 casi no cambia. `sublinear_tf` tampoco mueve el resultado de forma relevante. El cuello de botella no era el tamaño del vocabulario, sino las stopwords y el suavizado.

**`alpha` en Multinomial (0.644 → 0.678 con 0.1; 0.673 con 0.01).** `alpha` suaviza las probabilidades. Con el default 1.0, palabras muy típicas de una clase (`hockey`, `nasa`, `emacs`) se aplanan. Bajar a 0.1 permite que esas señales pesen más y el F1-macro sube. En 0.01 el valor baja un poco: el modelo se ajusta a rarezas de train que no se repiten en test.

**Multinomial vs Complement.** Con los mismos vectores, ComplementNB obtiene **0.692** y Multinomial 0.644 (0.678 si se baja `alpha`). Complement utiliza también las palabras que no pertenecen a la clase y corrige el sesgo hacia clases más grandes. El F1-macro penaliza justo las clases que salen peor, por eso Complement mejora. En este dataset el `alpha=1` de Complement fue mejor que 0.1 y 0.01.

**TF-IDF vs conteos.** El mismo Complement con `CountVectorizer` baja a 0.639. El IDF reduce el peso de términos muy frecuentes. Sin IDF, esas palabras vuelven a ensuciar el modelo, de forma parecida a no sacar stopwords.

**Conclusión.** El default del enunciado es un piso. El F1-macro mejora sobre todo al sacar stopwords, al usar ComplementNB y, en Multinomial, al bajar un poco `alpha`. La mejor configuración observada: TF-IDF con `stop_words='english'`, `min_df=2`, `max_df=0.5`, `sublinear_tf=True` y **ComplementNB** → **0.692**.


### 4. Similaridad entre palabras

Al transponer `X_train` se obtiene la matriz término-documento: cada palabra queda representada por los documentos en los que aparece y por su peso TF-IDF. Dos palabras son similares si ocurren en los mismos posts.

Se eligen 5 palabras a mano, como indica la consigna. Se incluye `car` porque aparece en el tutorial; las otras cubren temas distintos del corpus.


In [27]:
X_terms = X_train.T
print("Shape término-documento:", X_terms.shape, "(términos x documentos)")

palabras = ["car", "space", "god", "hockey", "encryption"]

for w in palabras:
    idx = tfidfvect.vocabulary_[w]
    cossim = cosine_similarity(X_terms[idx], X_terms)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]
    print("=" * 60)
    print("Palabra:", w)
    for i in mostsim:
        print(f"  {idx2word[i]:20s}  sim={cossim[i]:.3f}")


Shape término-documento: (101631, 11314) (términos x documentos)
Palabra: car
  cars                  sim=0.180
  criterium             sim=0.177
  civic                 sim=0.175
  owner                 sim=0.169
  dealer                sim=0.168


Palabra: space
  nasa                  sim=0.330
  seds                  sim=0.297
  shuttle               sim=0.293
  enfant                sim=0.280
  seti                  sim=0.246


Palabra: god
  jesus                 sim=0.269
  bible                 sim=0.262
  that                  sim=0.256
  existence             sim=0.255
  christ                sim=0.251
Palabra: hockey
  ncaa                  sim=0.274
  nhl                   sim=0.265
  affiliates            sim=0.248
  xenophobes            sim=0.243
  sportschannel         sim=0.223


Palabra: encryption
  microcircuit          sim=0.340
  scrambles             sim=0.338
  accommodates          sim=0.338
  nistnews              sim=0.338
  2758                  sim=0.338


**Interpretación**

Cada palabra es un vector de 11.314 dimensiones (una por documento de train). La similaridad mide co-ocurrencia en documentos, no sinonimia.

**`car` → `cars`, `criterium`, `civic`, `owner`, `dealer`.**  
`cars`, `civic`, `dealer` y `owner` pertenecen al vocabulario de `rec.autos`. `criterium` es un vecino débil: es un término poco frecuente y unas pocas coincidencias bastan para subirlo en el ranking.

Conclusión: el núcleo es coherente; el quinto vecino es ruido de una palabra rara.

**`space` → `nasa`, `seds`, `shuttle`, `enfant`, `seti`.**  
`nasa`, `shuttle` y `seti` son términos esperables de `sci.space`. `seds` es una agrupación estudiantil de espacio, también razonable. `enfant` no aporta significado: co-ocurre en algún documento puntual.

Conclusión: si la palabra es muy de un newsgroup, los vecinos suelen ser temáticos. El extraño aparece porque el método no modela sentido, solo documentos compartidos.

**`god` → `jesus`, `bible`, `that`, `existence`, `christ`.**  
`jesus`, `bible` y `christ` son coherentes (religión / ateísmo). `that` es una stopword: entra porque el vectorizador es el default, el mismo que en el punto 1 acercaba posts por `the` o `you`.

Conclusión: el bloque temático es correcto; el vecino `that` confirma que no filtrar palabras vacías afecta también la similaridad entre términos.

**`hockey` → `ncaa`, `nhl`, `affiliates`, `xenophobes`, `sportschannel`.**  
`nhl`, `ncaa` y `sportschannel` son ligas y TV de hockey. `xenophobes` no se relaciona con el deporte; coincide en algún hilo.

Conclusión: 3 de 5 vecinos son interpretables. Los otros dos muestran el límite del método.

**`encryption` → `microcircuit`, `scrambles`, `accommodates`, `nistnews`, `2758`.**  
Los scores son casi iguales (~0.338). Esas palabras raras aparecen juntas en un comunicado (NIST / chip Clipper) y casi no salen en otro lado, así que sus vectores quedan parecidos. No son sinónimos de *encryption*; son compañeras de un mismo documento.

Conclusión: no funciona como sinonimia y sí como detector de co-ocurrencia extrema.

**Conclusión.**  
La matriz transpuesta recupera asociaciones temáticas (`car`–`dealer`, `space`–`nasa`, `god`–`bible`, `hockey`–`nhl`) y también artefactos (stopwords, términos raros, compañeros de un FAQ). Por eso la consigna pide elegir las palabras a mano.
